# DeepSORT Extended — Google Colab

Run the extended DeepSORT tracker with modern detectors and ReID on MOT Challenge sequences.

**Runtime:** GPU (T4). **Data:** MOT sequences on Google Drive at `MyDrive/MOT/`.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository and install dependencies
REPO_URL = "https://github.com/<your-user>/deep-sort-project.git"  # <-- replace with your repo
!git clone $REPO_URL deep-sort-project
%cd deep-sort-project
!pip install -q -r requirements.txt

In [ ]:
# 3. Mount Google Drive and set MOT data path
from google.colab import drive
drive.mount('/content/drive')

MOT_DIR = "/content/drive/MyDrive/MOT"  # should contain MOT15/train, MOT16/train, etc.
import os
assert os.path.isdir(MOT_DIR), "Upload MOT data to Google Drive: MyDrive/MOT/"

In [ ]:
# 4. Configuration — choose detector and ReID model
DETECTOR = "yolov8n"      # yolov8n | yolov5s | rtdetr_r18
REID = "osnet_x0_25"      # osnet_x0_25 | resnet50_ibn | fastreid_sbs
SEQUENCE = "MOT16-09"     # demo sequence
CONFIG = "configs/default.yaml"

# Resolve sequence path
for split in ["train", "test", ""]:
    if split:
        p = os.path.join(MOT_DIR, "MOT16", split, SEQUENCE)
    else:
        p = os.path.join(MOT_DIR, SEQUENCE)
    if os.path.isdir(p):
        SEQ_DIR = p
        break
print("Sequence dir:", SEQ_DIR)

In [ ]:
# 5. Run tracking on one sequence (demo)
!python run_tracker.py \
  --sequence_dir {SEQ_DIR} \
  --config {CONFIG} \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

In [ ]:
# 6. Generate overlay video
!python tools/generate_overlays.py \
  --mot_dir {os.path.dirname(os.path.dirname(SEQ_DIR))} \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video
Video("results/overlays/{SEQUENCE}_overlay.mp4", embed=True)

In [ ]:
# 7. Full evaluation on all 6 sequences (optional, takes ~1-2 hours)
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    !python eval/run_mot.py --mot_dir {MOT_DIR}/MOT16/train --output_dir results/modern
    !python eval/eval_detector.py --mot_dir {MOT_DIR}/MOT16/train --detector {DETECTOR}
    !python eval/eval_reid.py --mot_dir {MOT_DIR}/MOT16/train --reid {REID}

## MOT15 sequences

For TUD-Campus, TUD-Stadtmitte, KITTI-17, PETS09-S2L1 use `MOT_DIR/MOT15/train` instead of MOT16.